# Ranking Signal Analysis: Do Observable Search Signals Predict Content Decline?

- Author: Vuong Quoc Anh
- Lane: Ranking Signal Analysis
- Repo: https://github.com/AnhQuoc1234/flyrank-ml-internship-starter
- Date: 10/9/2026

## Abstract:
- Content teams managing large page inventories cannot 
manually review every page for decline risk so this project whether observable search signals including page position and impression volume which can help prioritize the review by using 9.8 million rows of real, pseudonymized search performance data from March 2026.

## Acknowledgments & Data Credit
- This work was built on the FlyRank ML Internship dataset, a pseudonymized warehouse release of real content and search performance data. Built on the FlyRank ML Internship dataset with following link below:
[flyrank.ai](https://flyrank.ai).


## 1. Problem Framing

- This project is helping a content review team prioritize which pages to look at first when review capacity is limited. The analysis unit is one content page (`content_hash_id`) on its first half of March 2026 signals. 

- The output is a ranked score with a reason code and an action tier (priority_review, review, monitor, monitor_low_priority). To do this, a human would focus on the flagged pages and move them to the top of their weekly queue instead of treating all pages equally.

- The cost of a wrong call is two-sided: false positives waste reviewer time on a page that didn’t need attention; false negatives let a real decline go unnoticed longer, costing more traffic the longer it is missed. Data/ML helps here as no reviewer can manually scan 500K+.

- Wrong calls are costly on both sides: a false positive wastes reviewer time on a page that didn't need it, and a false negative leaves a real decline undetected longer, costing more traffic the longer it is undetected . Data/ML helps with that as no reviewer can manually scan 500K+ content items for patterns at that scale – a ranked, evidence-based queue replaces guesswork with a defensible starting point, even if the underlying signal is modest.

## 2. Data Safety

- Data Used: `fact_content_daily_performance` partition of `month=2026-03` from FlyRank internship warehouse release (Hugging Face). 
March 2026 was intentionally selected as a mid-panel month – the last month (June 2026, the `_sample` table) is a sealed test month held out so as to not use the natural outcome window of any past→future label.

- Intentionally excluded: No FlyRank product decision outputs (`health_score`, `priority_score`, `action_type`) are present anywhere in this pipeline - they are not present in the released data, nor would they ever be used as model features if reconstructed, they are only external context.

- Potential for leakage: The label (`is_declining`) is calculated by comparing impressions in the second half of March with impressions in the first half. Only first half fields (impressions_first, avg_position_first) are used as features - confirmed by explicit audit (Week 6) that there is no label-derived or future-window column in the feature set. client_hash_id and content_hash_id are pseudo-anonymous identifiers only used for grouping (train/test split, deduplication) - not as model features.

- Public-safety confirmation: No client names, domains, URLs, or raw queries appear anywhere in this repo or report only pseudonymized hash IDs and aggregated metrics.

## 3. Baseline
- The baseline is a transparent rule: rank pages by `impressions_first` alone (first-half March volume), with no model involved. This was chosen after verifying two candidate signals directly in the data:

- `avg_position_first`: verdict and MIXED. Decline rate was roughly flat (33-35%) across most position buckets, dropping only at 50+ (~26%). No clean, usable relationship.

- `gsc_impressions` (volume): verdict and CONFIRMED. Decline rate rose monotonically from 24.5% (0-10 impressions) to 38.4% (2000+ impressions).

Because only volume held up under scrutiny, the baseline rule scores purely on `impressions_first`. On the held-out, client-grouped test split, this baseline scores **AUC = 0.5448**, the number every model in this report is compared against, on the same split and metric.

## 4. Model / Analysis

- Method: Decision Tree classifier (`max_depth=4`), chosen alongside with Logistic Regression as a comparison pair. Logistic Regression was included because it's the natural linear step up from a single-signal rule; the Decision Tree was added because Signal 1's bucket-shaped, non-monotonic pattern suggested a threshold-based method might capture 
structure a linear model couldn't.

- Features (2, both first-half/pre-decision-point):** `impressions_first`, `avg_position_first`. Second-half fields and any label-derived value were deliberately excluded.

- Target/proxy, in one sentence:** `is_declining` = whether a page's GSC impressions dropped more than 10% from the first half of March to the second half.

## 5. Evaluation

- Split: Client-grouped (`GroupShuffleSplit` on `client_hash_id`), 34 train clients / 9 test clients, zero overlap. This was chosen to deliver after the baseline playbook (Week 4) revealed real client concentration risk, a random split would let the model see pages from the same clients in both train and test, inflating apparent performance without proving it generalizes to a new client.

- Model vs. baseline (same split, same metric — ROC AUC):

| Method | AUC |
|---|---|
| Baseline (impressions_first only) | 0.5448 |
| Logistic Regression | 0.4962 |
| Logistic Regression (scaled) | 0.4962 |
| Decision Tree (max_depth=4) | 0.5511 |

- Before/after split comparison** (same Decision Tree, different splits):

| Split | AUC | Client overlap |
|---|---|---|
| Random (naive) | 0.5578 | 41 of 41 clients in both sets |
| Client-grouped (honest) | 0.5511 | 0 clients in both sets |

- Base rate: 34% of pages in this sample are labeled declining, a worth stating alongside any accuracy-style number, since a model predicting "stable" for everyone would already be right 66% of the time.

- Error analysis: At the default 0.5 threshold, the Decision Tree's confusion matrix shows it predicts "declining" for only 15 of 10,247 test pages, correctly catching just 7 of 3,646 actual decliners (recall ≈ 0.2%). The 64.4% headline accuracy is misleading and it comes from the modeldefaulting to "stable" almost everywhere. Five sampled false negatives showed probabilities clustered just below the 0.5 cutoff (0.34–0.37), plus a mix of real sharp declines, near-noise low-volume cases, and cases where position and impressions pointed in opposite directions.

## 6. Interpretation
Feature importance in the Decision Tree: `impressions_first` (0.887) dominates over `avg_position_first` (0.113) — consistent with the signal-check finding that only volume showed a real, monotonic relationship with decline.

The central, honest finding of this project is a **negative result**: none of the modeled methods meaningfully beat a simple one-signal baseline, and adding model complexity (Logistic Regression, Decision Tree) did not reliably improve ranking quality — Logistic Regression actually scored below random guessing. This is a valid and useful 
result: it means the two available first-half signals carry limited, mostly-redundant information about decline, and effort spent on model complexity would be better spent finding new signals or redefining the label (e.g., a longer or different decline window) than tuning these 
two.

## 7. Recommendation
Based on `impressions_first` and `decline_prob`, pages are grouped into four action tiers:

- priority_review (10,423 pages, a  high volume, high estimated risk) 

- review (31,383, a high volume), monitor (77,249, standard volume), and monitor_low_priority (22,412 — very ow volume, signal close)

### Employer Summary
- During this internship, I have built validated a content-decline prioritization model on 9.8M rows of real search performance data by using lient-grouped validation to avoid leakage and a rigorous baseline comparison to test.

- The result was doing great and well-supported negative finding. The model only marginally outperformed the baseline and had poor recall at a standard threshold, which turning into transparent, human review action playbook rather than overselling a weak signal. 

- One more time, this project reflects my pproach to ML work generally including rigorous validation, honest reporting of 
limitations, and decision-support framing over automation.

In [ ]:
# ============================================
# FULL SETUP — run this first, every time
# ============================================
import os
import pandas as pd
import duckdb
from huggingface_hub import hf_hub_download
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# 1. Load data
path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)
con = duckdb.connect()

df = con.execute(f"""
    WITH daily AS (
        SELECT
            content_hash_id, client_hash_id, report_date,
            gsc_impressions, gsc_avg_position, gsc_clicks,
            CASE WHEN report_date <= DATE '2026-03-15' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{path}')
        WHERE gsc_data_available IS TRUE
    ),
    agg AS (
        SELECT content_hash_id, client_hash_id, period,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_clicks) AS clicks,
               AVG(gsc_avg_position) AS avg_position
        FROM daily GROUP BY 1,2,3
    )
    SELECT
        f.content_hash_id, f.client_hash_id,
        f.impressions AS impressions_first, s.impressions AS impressions_second,
        f.avg_position AS avg_position_first, s.avg_position AS avg_position_second
    FROM agg f
    JOIN agg s ON f.content_hash_id = s.content_hash_id AND f.client_hash_id = s.client_hash_id
    WHERE f.period = 'first_half' AND s.period = 'second_half'
""").df()

# 2. Label
df["pct_change_impressions"] = (df["impressions_second"] - df["impressions_first"]) / df["impressions_first"].replace(0, pd.NA)
df["is_declining"] = (df["pct_change_impressions"] < -0.10).astype(int)

feature_cols = ["impressions_first", "avg_position_first"]

# 3. Client-grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))
train, test = df.iloc[train_idx], df.iloc[test_idx]

# 4. Baseline
baseline_auc = roc_auc_score(test["is_declining"], test["impressions_first"])

# 5. Logistic Regression
logreg = LogisticRegression(max_iter=1000)
logreg.fit(train[feature_cols], train["is_declining"])
logreg_auc = roc_auc_score(test["is_declining"], logreg.predict_proba(test[feature_cols])[:, 1])

# 6. Decision Tree (grouped split)
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(train[feature_cols], train["is_declining"])
tree_auc = roc_auc_score(test["is_declining"], tree.predict_proba(test[feature_cols])[:, 1])

# 7. Decision Tree (random split, for comparison)
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    df[feature_cols], df["is_declining"], test_size=0.2, random_state=42
)
tree_rand = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_rand.fit(X_train_rand, y_train_rand)
rand_auc = roc_auc_score(y_test_rand, tree_rand.predict_proba(X_test_rand)[:, 1])

print("Baseline:", round(baseline_auc, 4))
print("Logistic Regression:", round(logreg_auc, 4))
print("Decision Tree (grouped):", round(tree_auc, 4))
print("Decision Tree (random split):", round(rand_auc, 4))

In [1]:
#Chart Generation for capstone
import matplotlib.pyplot as plt

methods = ["Baseline\n(impressions)", "Logistic\nRegression", "Decision Tree\n(grouped split)", "Decision Tree\n(random split)"]
scores = [baseline_auc, logreg_auc, tree_auc, rand_auc]

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(methods, scores, color=["#888888", "#c0392b", "#2980b9", "#95a5a6"])
ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8, label="Random guessing")
ax.set_ylabel("ROC AUC")
ax.set_title("Model vs. Baseline — March 2026, Client-Grouped Test Split")
ax.set_ylim(0.4, 0.65)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, score + 0.005, f"{score:.4f}", ha="center", fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig("work/figures/model_vs_baseline_auc.png", dpi=150)
plt.show()


NameError: name 'baseline_auc' is not defined